# 实验一：静态页面与 API 数据采集

平台：SmartPark DataLab（https://115.159.159.82）

- Device：`/devices` → 动态加载 → API → JSON → 分页采集全部
- Company：`/companies` → 静态加载 → HTML → BeautifulSoup → "下一页" → 采集 3 页

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter
from pathlib import Path
import json

BASE = "https://115.159.159.82"

## 任务 1：Device —— 判断动态加载并找到 API

浏览器里右键查看 `/devices` 源代码，搜不到任何设备名（只有 `正在加载…`），说明数据是 JS 后加载的。
打开 F12 → Network → Fetch/XHR 可以看到请求：

```
GET https://115.159.159.82/api/devices?page=1&page_size=50
```

In [2]:
api_url = BASE + "/api/devices"

r = requests.get(api_url, params={"page": 1, "page_size": 50}, timeout=10)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [3]:
data = r.json()

print(type(data))
print("keys:", list(data.keys()))
print("page:", data["page"])
print("page_size:", data["page_size"])
print("total:", data["total"])
print("本页条数:", len(data["items"]))

<class 'dict'>
keys: ['page', 'page_size', 'total', 'items']
page: 1
page_size: 50
total: 5000
本页条数: 50


In [4]:
# 看一条真实设备记录
print(json.dumps(data["items"][0], ensure_ascii=False, indent=2))

{
  "device_id": "D00000001",
  "company_id": "C000001",
  "device_name": "循环泵-000001-001",
  "device_type": "pump",
  "model": "SP-PUM-800",
  "zone": "智造一区",
  "install_date": "2021-09-19",
  "rated_power_kw": 71.2,
  "sampling_interval_s": 30,
  "protocol": "mqtt",
  "status": "running",
  "updated_at": "2026-09-01T03:58:00"
}


## 任务 2：Device API 分页采集

分页字段：`page`（当前页）、`page_size`（每页条数）、`total`（总数）、`items`（本页列表）。
总页数由 `total` 和 `page_size` 计算，**不写死**。

In [5]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(api_url, params={"page": page, "page_size": page_size}, timeout=10)
    page_data = r.json()
    devices.extend(page_data["items"])

print("device count:", len(devices))
print("unique device_id:", len(set(d["device_id"] for d in devices)))

device count: 5000
unique device_id: 5000


## 任务 3：Company —— 静态 HTML 解析

`/companies` 的源代码里能直接搜到企业名称，属于静态加载。

企业记录 DOM：

```html
<article class="company-card" data-company-id="C000001">
  <div class="company-card-top"><span class="mono">C000001</span><span class="badge active">运营中</span></div>
  <h2><a href="/companies/C000001">临港精工001有限公司</a></h2>
  <dl>
    <div><dt>所属行业</dt><dd>汽车零部件</dd></div>
    ...
    <div><dt>设备数量</dt><dd>6 台</dd></div>
  </dl>
</article>
```

selector 选 `article.company-card`：语义明确的 class，不依赖位置，页面插入新记录也不会失效。

In [7]:
r = requests.get(BASE + "/companies", timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")
print("本页企业数:", len(cards))

本页企业数: 20


In [8]:
# 先解析第一条，确认字段怎么取
card = cards[0]

company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one(".badge")["class"][-1]   # class 列表是 ['badge','active']，取最后一个
status_text = card.select_one(".badge").get_text(strip=True)

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print("company_id:", company_id)
print("name:", name)
print("status:", status, status_text)
print("details:", details)

company_id: C000001
name: 临港精工001有限公司
status: active 运营中
details: {'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


## 任务 4：Company "下一页"连续采集 3 页

分页导航：

```html
<nav class="pagination">
  <a href="/companies?page=1">上一页</a><b>2</b><a href="/companies?page=3">下一页</a>
</nav>
```

注意：第 1 页的"上一页"是 `<span>`，第 2 页起才变成 `<a>`。
所以不能用 `nav.pagination a`（会拿到"上一页"，导致 1↔2 反复横跳），要用 `a:last-of-type`。

In [9]:
next_link = soup.select_one("nav.pagination a:last-of-type")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [10]:
companies = []
current_url = BASE + "/companies"

for i in range(3):
    r = requests.get(current_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")
    cards = soup.select("article.company-card")

    for card in cards:
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)

        companies.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details.get("所属行业"),
            "zone": details.get("所在区域"),
            "device_count": int(details.get("设备数量", "0").replace("台", "").strip()),
            "status": card.select_one(".badge")["class"][-1],
            "status_text": card.select_one(".badge").get_text(strip=True),
        })

    print(f"第 {i + 1} 页: {len(cards)} 条")

    next_link = soup.select_one("nav.pagination a:last-of-type")
    current_url = urljoin(current_url, next_link.get("href"))

print("company count:", len(companies))
print("unique company_id:", len(set(c["company_id"] for c in companies)))

第 1 页: 20 条
第 2 页: 20 条


第 3 页: 20 条
company count: 60
unique company_id: 60


## 任务 5：结果验证

数量、ID 唯一性，以及 Company 页面 `device_count` 与 Device 实际统计的交叉一致性。

In [11]:
device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0
for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]
    actual = device_counts.get(cid, 0)
    if expected != actual:
        mismatch_count += 1

print("Company 总数:", len(companies))
print("唯一 company_id:", len(set(x["company_id"] for x in companies)))
print("Device 总数:", len(devices))
print("唯一 device_id:", len(set(x["device_id"] for x in devices)))
print("Company-Device 不一致企业数:", mismatch_count)

Company 总数: 60
唯一 company_id: 60
Device 总数: 5000
唯一 device_id: 5000
Company-Device 不一致企业数: 0


## 任务 6：保存采集结果

In [12]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
